In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split

In [2]:
train_data = pd.read_csv('/kaggle/input/catch-me-if-you-can-intruder-detection-through-webpage-session-tracking2/train_sessions.csv')
test_data = pd.read_csv('/kaggle/input/catch-me-if-you-can-intruder-detection-through-webpage-session-tracking2/test_sessions.csv')

# Подготовка данных и создание новых признаков

In [3]:
def prepare_data(df):
    # Копию можно не делать, если возвращаем новый df, но в Kaggle часто делают copy().
    data = df.copy()
    
    # Изменение типа данных object -> datetime, если похоже на время
    for col in data.select_dtypes(include=['object']).columns:
        # Попытка парсинга дат
        data[col] = pd.to_datetime(data[col], errors='ignore')
    
    # Все числовые (кроме session_id) столбцы -> object (видимо, это сайты)
    if 'session_id' in data.columns:
        numeric_cols = data.drop(['session_id'], axis=1).select_dtypes(include=['number']).columns
    else:
        numeric_cols = data.select_dtypes(include=['number']).columns
    
    for col in numeric_cols:
        data[col] = data[col].astype('object')
    
    # Обозначим категорику и время
    if 'session_id' in data.columns:
        categoric_features = data.drop(['session_id'], axis=1).select_dtypes(include=['object']).columns
    else:
        categoric_features = data.select_dtypes(include=['object']).columns
    
    time_features = data.select_dtypes(include=['datetime64[ns]']).columns
    
    # Заполняем пропуски: даты - NaT, категориальные - -1
    data[time_features] = data[time_features].fillna(pd.NaT)
    data[categoric_features] = data[categoric_features].fillna(-1).astype('object')
    
    # Вычисляем время сессии
    if len(time_features) > 0:
        data['session_time'] = data[time_features].max(axis=1) - data[time_features].min(axis=1)
    else:
        data['session_time'] = pd.Timedelta(0)  # на всякий случай
    
    # Кол-во переходов (сколько сайтов вообще посещено?)
    data['count_transition'] = data[categoric_features].count(axis=1)
    
    # Среднее время перехода
    if len(time_features) > 1:
        time_diffs = data[time_features].diff(axis=1).iloc[:, 1:]
        data['average_time_transition'] = time_diffs.sum(axis=1) / (time_diffs.notna().sum(axis=1))
    else:
        data['average_time_transition'] = pd.Timedelta(0)  # если только 1 временной признак, diff бессмыслен
    
    # Кол-во последовательных повторов
    def count_repeats(row):
        repeats = 1
        max_repeats = 1
        previous = None
        for site in row:
            if site == previous:
                repeats += 1
                max_repeats = max(max_repeats, repeats)
            else:
                repeats = 1
            previous = site
        return max_repeats
    
    data['max_repeats'] = data[categoric_features].apply(count_repeats, axis=1)
    
    # Стартовый и завершающий сайт (если хоть что-то есть)
    def first_non_null(row):
        return row.dropna().iloc[0] if any(row.dropna()) else -1

    def last_non_null(row):
        return row.dropna().iloc[-1] if any(row.dropna()) else -1
    
    data['start_site'] = data[categoric_features].apply(first_non_null, axis=1).astype('object')
    data['end_site'] = data[categoric_features].apply(last_non_null, axis=1).astype('object')
    
    # Начало и конец сессии
    if len(time_features) > 0:
        data['time_start_session'] = data[time_features].min(axis=1)
        data['time_end_session'] = data[time_features].max(axis=1)
    else:
        data['time_start_session'] = pd.to_datetime('1970-01-01')
        data['time_end_session'] = pd.to_datetime('1970-01-01')
    
    # День недели и выходные
    data['day_of_week'] = data['time_start_session'].dt.weekday
    data['is_weekend'] = (data['day_of_week'] >= 5).astype(int)
    
    # Время суток
    def classify_time_of_day(timestamp):
        if isinstance(timestamp, pd.Timestamp):
            hour = timestamp.hour
            if 5 <= hour < 12:
                return 'Morning'
            elif 12 <= hour < 17:
                return 'Daytime'
            elif 17 <= hour < 21:
                return 'Evening'
            else:
                return 'Night'
        return 'Unknown'
    
    data['session_start_period'] = data['time_start_session'].apply(classify_time_of_day)
    data['session_end_period'] = data['time_end_session'].apply(classify_time_of_day)
    
    # Плотность сессии
    data['session_density'] = (
        data['count_transition'] / data['session_time'].dt.total_seconds().replace(0, np.nan)
    ).replace([np.inf, -np.inf], 0).fillna(0)
    
    return data

In [4]:
# Применяем ко train
train = train_data.copy()
y = train['target']
train.drop(['target'], axis=1, inplace=True)

train_prepared = prepare_data(train)

# Модель

## Функции трансформации временных признаков и категориальных

In [5]:
def preprocess_datetime_columns(X):
    # datetime -> секунды
    for column in X.select_dtypes(include=[np.datetime64]).columns:
        X[column] = (
            X[column].astype('datetime64[s]') - pd.Timestamp("1970-01-01")
        ).dt.total_seconds()
        X[column].fillna(0, inplace=True)
    return X

def preprocess_timedelta_columns(X):
    # timedelta -> секунды
    for column in X.select_dtypes(include=[np.timedelta64]).columns:
        X[column] = X[column].dt.total_seconds()
        X[column].fillna(0, inplace=True)
    return X

def sites_to_text(X):
    """Склеить все значения (названия сайтов) в одну строку."""
    return X.apply(lambda row: ' '.join(row.astype(str)), axis=1)

## Сплитовка данных перед подачей в модель

In [6]:
X = train_prepared.drop('session_id', axis=1)  # Все, кроме session_id
# Переводим datetime/timedelta -> числа
X = preprocess_datetime_columns(X)
X = preprocess_timedelta_columns(X)

# Целевая переменная в бинарном формате
y_encoder = LabelEncoder()
y = y_encoder.fit_transform(y)

# Разделяем на train/test
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [7]:
# 4.1 Числовые колонки
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

# 4.2 Категориальные колонки
cat_cols = X_train.select_dtypes(include=['object']).columns

#   - выделим колонку(и) с сайтами
site_columns = [col for col in cat_cols if 'site' in col]
#   - остальные категориальные
other_cat_columns = list(set(cat_cols) - set(site_columns))

## LogisticRegression

In [8]:
# 5.1 Числовой пайплайн
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# 5.2 Пайплайн для сайтов (TF-IDF)
site_pipeline = Pipeline(steps=[
    ('sites_to_text', FunctionTransformer(sites_to_text)),
    ('tfidf', TfidfVectorizer())
])

# 5.3 Пайплайн для прочих категорий
other_cat_pipeline = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, num_cols),
    ('sites', site_pipeline, site_columns),
    ('other_cat', other_cat_pipeline, other_cat_columns),
], remainder='drop')  


model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        C=4,
        solver='liblinear',
        penalty='l1',
        random_state=42,
        max_iter=1000
    ))
])

model_pipeline.fit(X_train, y_train)


y_pred = model_pipeline.predict(X_val)
y_prob = model_pipeline.predict_proba(X_val)[:, 1]

fpr, tpr, _ = roc_curve(y_val, y_prob)
val_roc_auc = auc(fpr, tpr)
print(f"Validation ROC AUC = {val_roc_auc:.4f}")

Validation ROC AUC = 0.9780


# Submisson

In [9]:
test_prepared = prepare_data(test_data)
test_prepared.drop(['target'], axis=1, errors='ignore', inplace=True)  # если вдруг есть
X_test_ids = test_prepared['session_id']
X_test_sub = test_prepared.drop('session_id', axis=1)

# datetime/timedelta -> числа
X_test_sub = preprocess_datetime_columns(X_test_sub)
X_test_sub = preprocess_timedelta_columns(X_test_sub)

# Предсказание
test_probs = model_pipeline.predict_proba(X_test_sub)[:, 1]

submission = pd.DataFrame({
    'session_id': X_test_ids,
    'target': test_probs
})
submission.to_csv('submission.csv', index=False)
print("Submission saved!")

Submission saved!


In [10]:
submission

,session_id,target
0,1,2.302945e-06
1,2,1.675627e-06
2,3,7.716429e-07
3,4,1.372728e-09
4,5,1.708670e-04
...,...,...
82792,82793,6.871923e-10
82793,82794,2.575487e-04
82794,82795,2.605627e-05
82795,82796,1.366542e-06
